# Week 06 | Reproducible Data Pipeline
**Team 06 (TEAM-06) · Google Colab · English**

**Members:** Salem Alharbi (A) · Mohammed Almogbel (B) · Ziyad Alharbi (C) · Fahad Hamdan Alharbi (D)

**Input:** `week_06_team_project_data_S000.zip` from Google Drive. The source dataset is named **S000**; the submitting group is **Team 06**.

**Run:** In Colab, select **Runtime → Restart session and run all**.

**Workflow:** Intake → Cleaning → Integration → Analysis → Verified outputs.

## Setup — connect Google Drive
- Keep the original ZIP in **My Drive**; edit `ZIP_PATH` if it is in a folder.
- Extract the four input files before running the exercises.

In [ ]:
import io, json, hashlib, os, sqlite3, tempfile, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

STUDENT_ID = "TEAM-06"
INPUT_DIR = Path("/content/day05_team_files")
OUTPUT_DIR = Path("/content/team_output")
RAW_COLUMNS = ["request_id", "desk_code", "desk", "request_type", "status", "opened_at", "minutes"]
ACCEPTED_COLUMNS = ["source_row"] + RAW_COLUMNS
ANALYTICAL_COLUMNS = ["source", "source_row", "request_id", "desk_code", "desk", "request_type", "status", "opened_at", "minutes", "region", "staffed_hours", "month", "hours"]
REJECTION_RULES = ["conflicting_id", "unknown_desk", "bad_date", "implausible_minutes"]
MONTHS = ("jan", "feb", "mar")

### Original project data
- This notebook reads the **actual ZIP you uploaded**, not embedded or generated data.
- The ZIP must contain January and February CSVs, the March SQLite database, and `targets.csv`.

In [ ]:
from google.colab import drive
from google.colab import files

drive.mount("/content/drive")

# Locate the original data ZIP (also supports folders inside My Drive).
ZIP_NAME = "week_06_team_project_data_S000.zip"
drive_root = Path("/content/drive/MyDrive")
matches = sorted(drive_root.rglob(ZIP_NAME))
if matches:
    ZIP_PATH = matches[0]
else:
    print(f"{ZIP_NAME} was not found in Drive. Upload it from your device:")
    uploaded = files.upload()
    if ZIP_NAME not in uploaded:
        raise FileNotFoundError(f"Please select {ZIP_NAME}")
    ZIP_PATH = Path("/content") / ZIP_NAME

# Extract the required files to a fixed location, independent of Colab's working directory.
expected = {f"day05_team_files/{name}" for name in
            ("jan_requests.csv", "feb_requests.csv", "team_data.db", "targets.csv")}
with zipfile.ZipFile(ZIP_PATH) as archive:
    archived = set(archive.namelist())
    if archived != expected:
        raise ValueError(f"Unexpected ZIP contents: {sorted(archived)}")
    archive.extractall("/content")

missing = [name for name in sorted(expected) if not (Path("/content") / name).is_file()]
if missing:
    raise FileNotFoundError(f"These files were not extracted: {missing}")
print("Data ZIP:", ZIP_PATH)
print("Input folder:", INPUT_DIR)
print("Files:", sorted(p.name for p in INPUT_DIR.iterdir()))


## Team plan / interface contracts
- Each role returns a DataFrame with the agreed column types; invalid interfaces raise `ValueError`.
- Each member owns a distinct function and one test; integration happens after the role checks pass.

In [ ]:
TEAM_PLAN = {
    "A": {"owner": "Salem Alharbi", "tasks": ["EX01", "EX02"], "handoff": "raw_month"},
    "B": {"owner": "Mohammed Almogbel", "tasks": ["EX03", "EX04"], "handoff": "accepted"},
    "C": {"owner": "Ziyad Alharbi", "tasks": ["EX05", "EX06"], "handoff": "analytical"},
    "D": {"owner": "Fahad Hamdan Alharbi", "tasks": ["EX07", "EX08", "EX10"], "handoff": "figures and brief"},
    "TEAM": {"tasks": ["EX09"], "handoff": "verified outputs and manifest"},
}
CONTRIBUTIONS = {"A": ["load_sources", "intake_report"],
                 "B": ["clean_month", "clean_all_months"],
                 "C": ["integrate", "compare_targets"],
                 "D": ["summaries", "make_team_figures", "brief_facts"],
                 "TEAM": ["run_pipeline"]}

def check_interface(df, name):
    """Enforce text, number and date contracts at all three hand-off points."""
    required = {"raw_month": RAW_COLUMNS, "accepted": ACCEPTED_COLUMNS,
                "analytical": ANALYTICAL_COLUMNS}
    if name not in required or not isinstance(df, pd.DataFrame):
        raise ValueError(f"Unknown or non-DataFrame interface: {name}")
    if not df.columns.is_unique:
        raise ValueError(f"{name}: duplicate column names")
    missing = [c for c in required[name] if c not in df.columns]
    if missing:
        raise ValueError(f"{name}: missing columns {missing}")

    text_fields = RAW_COLUMNS if name == "raw_month" else [
        "request_id", "desk_code", "desk", "request_type", "status"]
    if name == "analytical":
        text_fields = ["source", *text_fields, "region"]
    for field in text_fields:
        series = df[field]
        if not (pd.api.types.is_string_dtype(series.dtype) or
                pd.api.types.is_object_dtype(series.dtype)):
            raise ValueError(f"{name}: {field} must be text")
        if not series.dropna().map(lambda value: isinstance(value, str)).all():
            raise ValueError(f"{name}: {field} contains non-text values")

    if name != "raw_month":
        if not pd.api.types.is_integer_dtype(df["source_row"]):
            raise ValueError(f"{name}: source_row must be an integer")
        if not pd.api.types.is_datetime64_any_dtype(df["opened_at"]):
            raise ValueError(f"{name}: opened_at must be datetime")
        if not pd.api.types.is_numeric_dtype(df["minutes"]):
            raise ValueError(f"{name}: minutes must be numeric")
    if name == "analytical":
        if not pd.api.types.is_numeric_dtype(df["hours"]):
            raise ValueError("analytical: hours must be numeric")
        if not pd.api.types.is_integer_dtype(df["month"]):
            raise ValueError("analytical: month must be an integer")
        if not pd.api.types.is_numeric_dtype(df["staffed_hours"]):
            raise ValueError("analytical: staffed_hours must be numeric")
    return True

## EX01 — `load_sources` (Role A · 10 points)
- Read CSV fields as text; query SQLite **read-only** with explicit columns.
- Convert targets to integers and preserve February's extra `channel` column.

In [ ]:
def load_sources(paths):
    p = {k: Path(v) for k, v in paths.items()}
    jan = pd.read_csv(p["jan"], dtype=str, keep_default_na=False)
    feb = pd.read_csv(p["feb"], dtype=str, keep_default_na=False)
    db_uri = f"file:{p['db'].resolve().as_posix()}?mode=ro"
    with sqlite3.connect(db_uri, uri=True) as con:
        mar = pd.read_sql_query(
            "SELECT request_id, desk_code, desk, request_type, status, opened_at, minutes FROM march_requests", con
        ).fillna("").astype(str)
        desks = pd.read_sql_query(
            "SELECT desk_code, region, staffed_hours FROM desks", con,
            dtype={"desk_code": str, "region": str, "staffed_hours": int}
        )
    targets = pd.read_csv(p["targets"], dtype={"region": str, "month": int, "target_requests": int})
    for table in (jan, feb, mar):
        check_interface(table, "raw_month")
    if not {"desk_code", "region", "staffed_hours"}.issubset(desks.columns):
        raise ValueError("desks: missing required columns")
    if not {"region", "month", "target_requests"}.issubset(targets.columns):
        raise ValueError("targets: missing required columns")
    return {"jan": jan, "feb": feb, "mar": mar, "desks": desks, "targets": targets}

PATHS = {"jan": INPUT_DIR / "jan_requests.csv", "feb": INPUT_DIR / "feb_requests.csv",
         "db": INPUT_DIR / "team_data.db", "targets": INPUT_DIR / "targets.csv"}
sources = load_sources(PATHS)
print("Raw rows:", {m: len(sources[m]) for m in MONTHS})

## EX02 — `intake_report` (Role A · 8 points)
- Report raw row counts, exact duplicates, missing-value tokens and extra columns.
- Check the raw data **before cleaning** so no issue is hidden.

In [ ]:
def intake_report(sources):
    rows = []
    tokens = {"", "unknown", "n/a"}
    for month in MONTHS:
        df = sources[month]
        vals = df[RAW_COLUMNS].apply(lambda col: col.astype(str).str.strip().str.lower().isin(tokens))
        rows.append({"source": month, "raw_rows": int(len(df)),
                     "exact_duplicates": int(df.duplicated().sum()),
                     "missing_tokens": int(vals.to_numpy().sum()),
                     "extra_columns": ",".join(c for c in df.columns if c not in RAW_COLUMNS)})
    return pd.DataFrame(rows)

intake = intake_report(sources)
display(intake)

## EX03 — `clean_month` (Role B · 12 points)
- Apply the course policy: exact duplicates first; normalize text/codes, parse minutes/dates and flag failures.
- Reject **all conflicting IDs**, unknown desks, invalid dates and out-of-range minutes; **keep missing minutes**.

In [ ]:
def clean_month(raw_month, valid_codes, max_minutes=480):
    check_interface(raw_month, "raw_month")
    raw = raw_month.copy(deep=True).reset_index(drop=True)
    raw.insert(0, "source_row", np.arange(1, len(raw) + 1))
    # Compare ALL original raw fields, including February's channel, before dropping extras.
    original_cols = [c for c in raw.columns if c != "source_row"]
    exact = raw.duplicated(subset=original_cols, keep="first")
    work = raw.loc[~exact, ACCEPTED_COLUMNS].copy()
    work["desk_code"] = work["desk_code"].str.strip().apply(lambda x: x.zfill(3) if x.isdigit() else x)
    work["desk"] = work["desk"].str.strip().str.title()
    work["request_type"] = work["request_type"].str.strip().str.title()
    work["status"] = work["status"].str.strip().str.lower()
    work["request_id"] = work["request_id"].str.strip()
    raw_minutes = work["minutes"].str.strip()
    suffix = raw_minutes.str.extract(r"^(\d+(?:\.\d+)?)\s*min$")[0]
    work["minutes"] = pd.to_numeric(raw_minutes.where(suffix.isna(), suffix), errors="coerce")
    work["opened_at"] = pd.to_datetime(work["opened_at"], format="%Y-%m-%d %H:%M", errors="coerce")
    flags = pd.DataFrame(index=work.index)
    flags["conflicting_id"] = work.duplicated("request_id", keep=False)
    flags["unknown_desk"] = ~work["desk_code"].isin(set(valid_codes))
    flags["bad_date"] = work["opened_at"].isna()
    flags["implausible_minutes"] = work["minutes"].notna() & ((work["minutes"] <= 0) | (work["minutes"] > max_minutes))
    bad = flags.any(axis=1)
    reason_text = flags.apply(lambda row: ";".join([n for n in REJECTION_RULES if bool(row[n])]), axis=1)
    accepted = work.loc[~bad, ACCEPTED_COLUMNS].reset_index(drop=True)
    rejected = work.loc[bad, ACCEPTED_COLUMNS].copy()
    rejected["reasons"] = reason_text.loc[bad]
    rejected = rejected.reset_index(drop=True)
    report = {"raw_rows": int(len(raw)), "exact_duplicates": int(exact.sum()),
              "accepted": int(len(accepted)), "rejected": int(len(rejected))}
    assert report["raw_rows"] == report["exact_duplicates"] + report["accepted"] + report["rejected"]
    check_interface(accepted, "accepted")
    return accepted, rejected, report

valid_codes = set(sources["desks"]["desk_code"].astype(str).str.strip())
accepted_jan, rejected_jan, report_jan = clean_month(sources["jan"], valid_codes)
print("January:", report_jan)

## EX04 — Cleaning reconciliation (Role B · 8 points)
- Run the **same cleaner** for all three sources and keep each accepted/rejected table.
- Prove `raw = exact duplicates + accepted + rejected` for every month.

In [ ]:
def clean_all_months(sources, valid_codes):
    cleaned, summary = {}, []
    for month in MONTHS:
        accepted, rejected, report = clean_month(sources[month], valid_codes)
        cleaned[month] = {"accepted": accepted, "rejected": rejected, "report": report}
        summary.append({"source": month, **report,
                        "reconciled": report["raw_rows"] == report["exact_duplicates"] + report["accepted"] + report["rejected"]})
    return cleaned, pd.DataFrame(summary)

cleaned, cleaning_summary = clean_all_months(sources, valid_codes)
display(cleaning_summary)
assert cleaning_summary["reconciled"].all()

## EX05 — `integrate` (Role C · 10 points)
- Concatenate accepted months, add `source`, then enrich using **validated many-to-one** desk lookup.
- Create `month` and `hours`, and return the agreed column order.

In [ ]:
def integrate(cleaned, desks):
    lookup = desks[["desk_code", "region", "staffed_hours"]].copy()
    lookup["desk_code"] = lookup["desk_code"].astype(str).str.strip()
    lookup["desk_code"] = lookup["desk_code"].apply(lambda x: x.zfill(3) if x.isdigit() else x)
    if lookup["desk_code"].duplicated().any():
        raise ValueError("Duplicate desk_code in lookup")
    if lookup[["desk_code", "region", "staffed_hours"]].isna().any().any():
        raise ValueError("Missing lookup values")
    frames = []
    for month in MONTHS:
        accepted = cleaned[month]["accepted"]
        check_interface(accepted, "accepted")
        frames.append(accepted.assign(source=month))
    combined = pd.concat(frames, ignore_index=True)
    if combined["request_id"].duplicated().any():
        raise ValueError("Accepted request_id duplicated across months")
    analytical = combined.merge(lookup, on="desk_code", how="left", validate="many_to_one", indicator=True)
    if not analytical["_merge"].eq("both").all():
        raise ValueError("Accepted records have unmatched desk codes")
    analytical = analytical.drop(columns="_merge")
    analytical["staffed_hours"] = pd.to_numeric(analytical["staffed_hours"], errors="raise")
    analytical["month"] = analytical["opened_at"].dt.month.astype(int)
    analytical["hours"] = analytical["minutes"] / 60
    analytical = analytical[ANALYTICAL_COLUMNS].sort_values(["request_id", "source_row"], kind="stable").reset_index(drop=True)
    check_interface(analytical, "analytical")
    return analytical

analytical = integrate(cleaned, sources["desks"])
print("Analytical records:", len(analytical)); display(analytical.head(3))

## EX06 — `compare_targets` (Role C · 10 points)
- Start from **all targets**, even where zero requests were accepted; validate one-to-one merge.
- Compute actual requests and `gap = actual_requests - target_requests`.

In [ ]:
def compare_targets(analytical, targets):
    check_interface(analytical, "analytical")
    goals = targets[["region", "month", "target_requests"]].copy()
    goals["month"] = goals["month"].astype(int)
    goals["target_requests"] = goals["target_requests"].astype(int)
    if goals.duplicated(["region", "month"]).any():
        raise ValueError("Duplicate region/month targets")
    actual = analytical.groupby(["region", "month"], as_index=False).size().rename(columns={"size": "actual_requests"})
    result = goals.merge(actual, how="left", on=["region", "month"], validate="one_to_one")
    result["actual_requests"] = result["actual_requests"].fillna(0).astype(int)
    result["gap"] = result["actual_requests"] - result["target_requests"]
    return result.sort_values(["region", "month"], kind="stable").reset_index(drop=True)

vs_target = compare_targets(analytical, sources["targets"])
display(vs_target)

## EX07 — `summaries` (Role D · 10 points)
- Produce a region-by-month pivot of **accepted request counts**.
- Summarize staffed desk volume and handling minutes without inventing missing values.

In [ ]:
def summaries(analytical):
    check_interface(analytical, "analytical")
    region_month = analytical.pivot_table(index="region", columns="month", values="request_id", aggfunc="count", fill_value=0)
    region_month = region_month.reindex(columns=[1, 2, 3], fill_value=0).astype(int).reset_index()
    region_month.columns = ["region", "month_1", "month_2", "month_3"]
    desks_summary = (analytical.groupby(["desk_code", "desk", "region"], as_index=False)
                     .agg(requests=("request_id", "size"), total_minutes=("minutes", "sum"),
                          mean_minutes=("minutes", "mean")))
    desks_summary = desks_summary.sort_values("desk_code", kind="stable").reset_index(drop=True)
    return region_month.sort_values("region").reset_index(drop=True), desks_summary

region_month, desks_summary = summaries(analytical)
display(region_month); display(desks_summary)

## EX08 — `make_team_figures` (Role D · 10 points)
- Compare **January, February and March** in all three charts: requests by region, handling minutes by desk, and actual vs target.
- Save three clear PNG figures with labels, legends and consistent month colors.

In [ ]:
def make_team_figures(analytical, desks_summary, vs_target):
    month_names = {1: "Jan", 2: "Feb", 3: "Mar"}
    month_colors = {1: "#4F63D9", 2: "#20B9B0", 3: "#F4AD53"}
    figures = {}

    def save(name, fig, auto_layout=True):
        if auto_layout:
            fig.tight_layout(pad=1.6)
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=150, facecolor="white",
                    metadata={"Software": "Matplotlib"})
        plt.close(fig)
        figures[name] = buf.getvalue()

    # 1. Accepted requests: regions side by side, three months per region.
    region_month = (analytical.groupby(["region", "month"]).size()
                    .unstack(fill_value=0).reindex(columns=[1, 2, 3], fill_value=0))
    regions = list(region_month.index)
    x = np.arange(len(regions))
    width = 0.24
    fig, ax = plt.subplots(figsize=(10, 5.0))
    for j, m in enumerate((1, 2, 3)):
        bars = ax.bar(x + (j - 1) * width, region_month[m], width,
                      color=month_colors[m], label=month_names[m])
        ax.bar_label(bars, padding=3, fontsize=9)
    ax.set_xticks(x, regions)
    ax.set(title="Accepted requests by region and month", xlabel="Region", ylabel="Accepted requests")
    ax.legend(title="Month", ncol=3)
    ax.grid(axis="y", alpha=0.18)
    ax.set_axisbelow(True)
    save("requests_by_region.png", fig)

    # 2. Handling minutes: stacked monthly contribution for each desk.
    desk_month = (analytical.pivot_table(index="desk_code", columns="month",
                  values="minutes", aggfunc="sum", fill_value=0)
                  .reindex(columns=[1, 2, 3], fill_value=0))
    desk_month = desk_month.sort_index()
    x = np.arange(len(desk_month))
    bottom = np.zeros(len(desk_month))
    fig, ax = plt.subplots(figsize=(10, 5.0))
    for m in (1, 2, 3):
        values = desk_month[m].to_numpy(dtype=float)
        ax.bar(x, values, bottom=bottom, color=month_colors[m],
               label=month_names[m], width=0.66)
        bottom += values
    ax.set_xticks(x, list(desk_month.index))
    ax.set(title="Total handling minutes by desk and month",
           xlabel="Desk code", ylabel="Handling minutes (sum)")
    ax.legend(title="Month", ncol=3)
    ax.grid(axis="y", alpha=0.18)
    ax.set_axisbelow(True)
    save("minutes_by_desk.png", fig)

    # 3. Actual vs target: one panel for each month, grouped by region.
    fig, axes = plt.subplots(1, 3, figsize=(12, 4.8), sharey=True)
    for ax, m in zip(axes, (1, 2, 3)):
        d = vs_target.loc[vs_target["month"] == m].sort_values("region")
        xx = np.arange(len(d))
        bars_actual = ax.bar(xx - 0.18, d["actual_requests"], 0.36,
                             color="#4F63D9", label="Actual")
        bars_target = ax.bar(xx + 0.18, d["target_requests"], 0.36,
                             color="#9DAABD", label="Target")
        ax.bar_label(bars_actual, padding=2, fontsize=8)
        ax.bar_label(bars_target, padding=2, fontsize=8)
        ax.set_xticks(xx, d["region"])
        ax.set(title=month_names[m], xlabel="Region")
        ax.grid(axis="y", alpha=0.18)
        ax.set_axisbelow(True)
    axes[0].set_ylabel("Accepted requests")
    fig.suptitle("Actual requests vs targets | by region and month", fontsize=14, y=0.98)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=2,
               bbox_to_anchor=(0.5, 0.91), frameon=False)
    fig.subplots_adjust(left=0.07, right=0.985, bottom=0.13, top=0.75, wspace=0.12)
    save("gap_vs_target.png", fig, auto_layout=False)  # Preserve output filename.
    return figures

figures = make_team_figures(analytical, desks_summary, vs_target)
print("Charts:", list(figures))

## EX09 — `run_pipeline` (Team · 12 points)
- **Validate first**, then generate four CSVs, three PNGs and a manifest in stable order.
- Verify byte-identical reruns; an invalid desk lookup must raise `ValueError` **before writing**.

In [ ]:
def _csv_bytes(df):
    return df.to_csv(index=False, lineterminator="\n", date_format="%Y-%m-%d %H:%M", float_format="%.6f", na_rep="").encode("utf-8")

def run_pipeline(paths, out_dir, valid_codes=None):
    data = load_sources(paths)
    lookup_codes = set(data["desks"]["desk_code"].astype(str).str.strip())
    lookup_codes = {c.zfill(3) if c.isdigit() else c for c in lookup_codes}
    if valid_codes is not None and {str(c).strip().zfill(3) if str(c).strip().isdigit() else str(c).strip() for c in valid_codes} != lookup_codes:
        raise ValueError("valid_codes does not match the authoritative desks lookup")
    # Critical preflight, before output directory creation.
    raw_lookup = data["desks"]["desk_code"].astype(str).str.strip()
    normalized_lookup = raw_lookup.apply(lambda c: c.zfill(3) if c.isdigit() else c)
    if normalized_lookup.duplicated().any():
        raise ValueError("Duplicate desk_code in SQLite lookup")
    cleaned, summary = clean_all_months(data, lookup_codes)
    table = integrate(cleaned, data["desks"])
    comparison = compare_targets(table, data["targets"])
    by_region, by_desk = summaries(table)
    images = make_team_figures(table, by_desk, comparison)
    files = {"analytical_table.csv": _csv_bytes(table),
             "region_month.csv": _csv_bytes(by_region),
             "vs_target.csv": _csv_bytes(comparison),
             "cleaning_summary.csv": _csv_bytes(summary), **images}
    manifest = {
        "student_id": STUDENT_ID,
        "raw_rows": int(summary["raw_rows"].sum()),
        "exact_duplicates": int(summary["exact_duplicates"].sum()),
        "accepted": int(summary["accepted"].sum()),
        "rejected": int(summary["rejected"].sum()),
        "reconciled": bool(summary["reconciled"].all()),
        "source_rows": {r["source"]: int(r["raw_rows"]) for r in summary.to_dict("records")},
        "files_sha256": {k: hashlib.sha256(v).hexdigest() for k, v in sorted(files.items())},
    }
    files["manifest.json"] = (json.dumps(manifest, indent=2, sort_keys=True) + "\n").encode("utf-8")
    # All validation is complete; write only now.
    destination = Path(out_dir)
    destination.mkdir(parents=True, exist_ok=True)
    for name in sorted(files):
        (destination / name).write_bytes(files[name])
    return {"manifest": manifest, "files": files, "cleaned": cleaned,
            "analytical": table, "vs_target": comparison, "region_month": by_region,
            "cleaning_summary": summary, "desks_summary": by_desk}

result = run_pipeline(PATHS, OUTPUT_DIR)
print(json.dumps(result["manifest"], indent=2))

### Mandatory verification
- Re-run the pipeline and compare every output byte; reconcile all source rows.
- Deliberately duplicate a SQLite desk in a separate copy: the pipeline must fail without creating output files.

In [ ]:
import shutil
first_hashes = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in OUTPUT_DIR.iterdir() if p.is_file()}
rerun = run_pipeline(PATHS, OUTPUT_DIR)
second_hashes = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in OUTPUT_DIR.iterdir() if p.is_file()}
assert first_hashes == second_hashes, "Rerun changed output bytes"
assert result["manifest"]["reconciled"]
assert result["manifest"]["raw_rows"] == result["manifest"]["exact_duplicates"] + result["manifest"]["accepted"] + result["manifest"]["rejected"]
assert len(first_hashes) == 8
# Confirm the missing-minutes rule on an explicit valid request.
missing_sample = pd.DataFrame([{ "request_id": "TEST-MISSING", "desk_code": "001",
    "desk": "North", "request_type": "Report", "status": "open",
    "opened_at": "2025-01-01 10:00", "minutes": "unknown"}])
miss_acc, miss_rej, _ = clean_month(missing_sample, set(sources["desks"]["desk_code"].astype(str)))
assert len(miss_acc) == 1 and len(miss_rej) == 0 and pd.isna(miss_acc.loc[0, "minutes"])

# Contract regression checks: incorrect types must raise ValueError.
for field in ("desk_code", "request_id"):
    broken = cleaned["jan"]["accepted"].copy()
    broken[field] = 7
    try:
        check_interface(broken, "accepted")
        raise AssertionError(f"Incorrect {field} type was accepted")
    except ValueError:
        pass
broken = result["analytical"].copy()
broken["staffed_hours"] = "not numeric"
try:
    check_interface(broken, "analytical")
    raise AssertionError("Incorrect staffed_hours type was accepted")
except ValueError:
    pass
with tempfile.TemporaryDirectory() as temp:
    db_copy = Path(temp) / "broken.db"
    shutil.copy2(PATHS["db"], db_copy)
    with sqlite3.connect(db_copy) as conn:
        conn.execute("INSERT INTO desks SELECT * FROM desks LIMIT 1")
        conn.commit()
    bad_paths = dict(PATHS, db=db_copy)
    bad_out = Path(temp) / "should_not_exist"
    try:
        run_pipeline(bad_paths, bad_out)
        raise AssertionError("Duplicate desk lookup was accepted")
    except ValueError as e:
        assert "Duplicate desk_code" in str(e)
    assert not bad_out.exists(), "Output created despite failed preflight"
print("PASS: 8 outputs; byte-identical rerun; reconciliation; type contracts; missing durations; safe failure")

## EX10 — `brief_facts` + contributions (Role D + Team · 10 points)
- Generate facts **from actual outputs**, then provide a 100–180-word brief with two limitations.
- Record each role's contribution and a concise 100–180-word findings brief.

In [ ]:
def brief_facts(result):
    summary = result["cleaning_summary"]
    table = result["analytical"]
    vs = result["vs_target"]
    totals = summary[["raw_rows", "exact_duplicates", "accepted", "rejected"]].sum()
    by_region = table.groupby("region").size().sort_values(ascending=False)
    largest = by_region.index[0] if len(by_region) else "none"
    missing_minutes = int(table["minutes"].isna().sum())
    above = int((vs["gap"] > 0).sum())
    below = int((vs["gap"] < 0).sum())
    facts = {"raw_rows": int(totals["raw_rows"]), "exact_duplicates": int(totals["exact_duplicates"]),
             "accepted": int(totals["accepted"]), "rejected": int(totals["rejected"]),
             "top_region": largest, "top_region_requests": int(by_region.iloc[0]) if len(by_region) else 0,
             "missing_minutes_accepted": missing_minutes, "above_target_pairs": above, "below_target_pairs": below}
    brief = (
        f"This reproducible service-request pipeline processed {facts['raw_rows']} raw records across January, February and March. "
        f"The cleaning policy removed {facts['exact_duplicates']} exact duplicate rows, accepted {facts['accepted']} records, "
        f"and rejected {facts['rejected']} records under documented quality rules. All monthly counts reconcile. "
        f"The {facts['top_region']} region had the largest accepted volume, with {facts['top_region_requests']} requests. "
        f"Among the nine region-month comparisons, {facts['above_target_pairs']} exceeded their targets and "
        f"{facts['below_target_pairs']} fell below their targets. {facts['missing_minutes_accepted']} accepted requests "
        "have missing handling durations; these values were retained as missing rather than guessed or silently discarded. "
        "The output includes a validated analytical table, a regional pivot, a targets comparison, a cleaning summary and three charts. "
        "A manifest records the result totals and file checksums, allowing byte-for-byte checks after a repeated run. "
        "Two limitations matter: the data is synthetic, so it does not establish real police service performance, "
        "and the course-defined handling-time ceiling can misclassify legitimate unusually long cases. "
        "We recommend reviewing conflicting request identifiers and invalid timestamps with the data owner before using these results for decisions."
    )
    assert 100 <= len(brief.split()) <= 180, len(brief.split())
    return facts, brief

facts, BRIEF = brief_facts(result)
print("Facts:", facts)
print("\nBrief (", len(BRIEF.split()), "words):\n", BRIEF)

# Local contribution evidence only. The original course notebook's auto-grader
# is required to create or verify the official skill-score report.
ROLE_SKILLS = {"A": ["S1"], "B": ["S2"], "C": ["S3"], "D": ["S4", "S6"]}
SKILL_REPORT = {"team": STUDENT_ID,
    "shared_skill": "S5",
    "official_autograder_verified": False,
    "contributions": [
        {"role": role, "owner": TEAM_PLAN[role]["owner"],
         "skills": ROLE_SKILLS[role], "functions": funcs,
         "status": "Local checks passed; official auto-grade not available"}
        for role, funcs in CONTRIBUTIONS.items() if role != "TEAM"
    ]}
Path(f"results_W06-D5-B_{STUDENT_ID}.json").write_text(json.dumps(SKILL_REPORT, indent=2) + "\n", encoding="utf-8")
assert all(callable(globals()[fn]) for names in CONTRIBUTIONS.values() for fn in names)

## Bonus Analysis — SQL-Based Decision Insights (Mohammed Almogbel)
- These **optional analyses run after EX01–EX10** on the accepted analytical table. The required **4 CSV + 3 PNG + manifest** remain unchanged.
- SQL results and two additional figures are saved separately in `bonus_output/`; the findings are exploratory and the dataset is synthetic.

In [ ]:
# Bonus A/B — SQL queries on the validated analytical table; original SQLite is untouched.
BONUS_DIR = Path("/content/bonus_output")
BONUS_DIR.mkdir(parents=True, exist_ok=True)

bonus_conn = sqlite3.connect(":memory:")
result["analytical"].to_sql("analytical", bonus_conn, index=False, if_exists="replace")

handling_sql = """
SELECT request_type, month,
       COUNT(*) AS accepted_requests,
       COUNT(minutes) AS known_duration_count,
       ROUND(AVG(minutes), 2) AS avg_minutes
FROM analytical
GROUP BY request_type, month
ORDER BY request_type, month
"""
quality_sql = """
SELECT region, month,
       COUNT(*) AS accepted_requests,
       SUM(CASE WHEN minutes IS NULL THEN 1 ELSE 0 END) AS missing_minutes,
       ROUND(100.0 * SUM(CASE WHEN minutes IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS missing_pct
FROM analytical
GROUP BY region, month
ORDER BY region, month
"""

handling_sql_result = pd.read_sql_query(handling_sql, bonus_conn)
quality_sql_result = pd.read_sql_query(quality_sql, bonus_conn)
bonus_conn.close()

assert handling_sql_result["accepted_requests"].sum() == len(result["analytical"])
assert quality_sql_result["accepted_requests"].sum() == len(result["analytical"])
assert quality_sql_result["missing_minutes"].sum() == result["analytical"]["minutes"].isna().sum() == 36
assert handling_sql_result["known_duration_count"].sum() == result["analytical"]["minutes"].notna().sum()
handling_sql_result.to_csv(BONUS_DIR / "handling_by_type_month.csv", index=False)
quality_sql_result.to_csv(BONUS_DIR / "missing_minutes_by_region_month.csv", index=False)
print("Bonus A — request handling durations:")
display(handling_sql_result)
print("Bonus B — duration completeness:")
display(quality_sql_result)


In [ ]:
# Two optional charts. No change to the three mandatory EX08 charts.
month_labels = {1: "Jan", 2: "Feb", 3: "Mar"}
series_colors = {"Traffic": "#4B86CA", "Lost Item": "#0BAA9D", "Report": "#9CA9BD", "Document": "#DE9D58"}
fig, ax = plt.subplots(figsize=(10.8, 5.0))
for category in ["Traffic", "Lost Item", "Report", "Document"]:
    rows = handling_sql_result[handling_sql_result["request_type"] == category].sort_values("month")
    ax.plot(rows["month"], rows["avg_minutes"], marker="o", markersize=8,
            linewidth=2.8 if category in ("Traffic", "Lost Item") else 1.8,
            color=series_colors[category], label=category)
    if category in ("Traffic", "Lost Item"):
        feb_point = rows.loc[rows["month"] == 2, "avg_minutes"]
        if not feb_point.empty:
            ax.annotate(f"{feb_point.iloc[0]:.1f}", (2, feb_point.iloc[0]),
                        xytext=(0, 9), textcoords="offset points",
                        fontsize=10, ha="center", weight="bold")
ax.set_xticks([1, 2, 3], ["January", "February", "March"])
ax.set_ylim(17, 37.5)
ax.set_ylabel("Mean recorded handling minutes (not total requests)")
ax.set_title("Bonus A | Which request types changed in February?", weight="bold", loc="left", pad=16)
ax.grid(axis="y", alpha=.19)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(ncol=4, loc="lower center", bbox_to_anchor=(.5, -0.24), frameon=False)
fig.subplots_adjust(left=.10, right=.97, top=.85, bottom=.25)
fig.savefig(BONUS_DIR / "handling_time_by_request_type.png", dpi=170, facecolor="white",
            metadata={"Software": "Matplotlib"})
plt.show(); plt.close(fig)

regions = sorted(quality_sql_result["region"].unique())
heat = (quality_sql_result.pivot(index="region", columns="month", values="missing_pct")
        .reindex(index=regions, columns=[1, 2, 3]))
missing_ct = quality_sql_result.set_index(["region", "month"])
fig, ax = plt.subplots(figsize=(10, 4.6))
from matplotlib.colors import LinearSegmentedColormap
palette = LinearSegmentedColormap.from_list("quality", ["#D8F1E9", "#FCE7B0", "#E9A7A1"])
im = ax.imshow(heat.to_numpy(), cmap=palette, vmin=0, vmax=12, aspect="auto")
ax.set_xticks([0, 1, 2], ["January", "February", "March"])
ax.set_yticks(range(len(regions)), regions)
for i, region in enumerate(regions):
    for j, month in enumerate([1, 2, 3]):
        d = missing_ct.loc[(region, month)]
        ax.text(j, i, f"{d['missing_pct']:.1f}%\n{int(d['missing_minutes'])}/{int(d['accepted_requests'])} missing",
                ha="center", va="center", fontsize=12, weight="bold", color="#23344B")
ax.set_title("Bonus B | Missing handling minutes by region and month", weight="bold", loc="left", pad=16)
for sp in ax.spines.values(): sp.set_visible(False)
fig.subplots_adjust(left=.17, right=.96, top=.84, bottom=.17)
fig.savefig(BONUS_DIR / "missing_minutes_heatmap.png", dpi=170, facecolor="white",
            metadata={"Software": "Matplotlib"})
plt.show(); plt.close(fig)

jan = handling_sql_result.query("month == 1").set_index("request_type")
feb = handling_sql_result.query("month == 2").set_index("request_type")
traffic_change = 100 * (feb.loc["Traffic", "avg_minutes"] / jan.loc["Traffic", "avg_minutes"] - 1)
lost_change = 100 * (feb.loc["Lost Item", "avg_minutes"] / jan.loc["Lost Item", "avg_minutes"] - 1)
metro_missing = int(quality_sql_result.query("region == 'Metro'")["missing_minutes"].sum())
print(f"Traffic: {traffic_change:+.1f}% February vs January; Lost Item: {lost_change:+.1f}%")
print(f"Missing durations: 36 of 521 (Metro: {metro_missing}). These are signals to review, not proof of causes.")


## Final submission
- Keep `week_06_team_project_data_S000.zip` in Google Drive; run every cell from the beginning.
- Submit this notebook, the `team_output` directory (**4 CSV, 3 PNG (monthly comparisons), 1 manifest**) and the Team 06 skill-report JSON.
- The last cell also saves `Week06_Team06_Results.zip` in Google Drive.
- Hidden grading still requires the original Day-5 course notebook and its interface checks.
- `bonus_output/` contains two **optional** SQL result CSVs and charts for Mohammed’s extra insights; these do not replace official outputs.

In [ ]:
print("Deliverables:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" -", p.name)
report_path = Path(f"results_W06-D5-B_{STUDENT_ID}.json")
print(" -", report_path.name)

results_zip = Path("/content/drive/MyDrive/Week06_Team06_Results.zip")
with zipfile.ZipFile(results_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for p in sorted(OUTPUT_DIR.iterdir()):
        archive.write(p, arcname=f"team_output/{p.name}")
    archive.write(report_path, arcname=report_path.name)
    for p in sorted(BONUS_DIR.iterdir()):
        archive.write(p, arcname=f"bonus_output/{p.name}")
print("Saved to Google Drive:", results_zip)
print("Extra SQL charts:", sorted(p.name for p in BONUS_DIR.iterdir()))
from IPython.display import Image
for chart_name in ("requests_by_region.png", "minutes_by_desk.png", "gap_vs_target.png"):
    print("\n" + chart_name)
    display(Image(filename=str(OUTPUT_DIR / chart_name)))

## AI Project — Anomaly Detection vs Cleaning Rules (Team 06)
- **Question:** the cleaning policy rejects any request above a fixed **480-minute** ceiling. Can a model that learns what is *normal for each request type* find suspicious durations that the fixed rule misses?
- **Method:** an unsupervised **Isolation Forest** is compared with a simple robust baseline (median / MAD per request type) and with the fixed rule.
- **Scope:** this section runs **after EX01–EX10 and the SQL bonus**, reuses the pipeline's own `cleaned` tables, and writes only to `ai_output/`. The required **4 CSV + 3 PNG + manifest** are unchanged.
- **Honesty rules:** fixed random seeds, a baseline next to the model, and the brief is generated from the measured numbers. The dataset is synthetic, so the result shows a method, not real service performance.

### AI-1 — Modelling table and two detectors
- Use accepted rows plus rows rejected **only** for `implausible_minutes`; rows with missing durations cannot be scored and are counted, not guessed.
- Both detectors return a score (higher = more unusual) and a flag.

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score, precision_score, recall_score

AI_DIR = Path("/content/ai_output")
AI_SEED = 6                 # fixed seed: Team 06
AI_CONTAMINATION = 0.05     # share of rows the forest is allowed to flag
MAD_THRESHOLD = 3.5         # standard cut-off for the robust z-score
MAX_MINUTES = 480           # the course ceiling used by clean_month

def build_model_table(cleaned):
    """Accepted rows + rows the rule rejected only for implausible minutes; known durations only."""
    frames = []
    for month in MONTHS:
        accepted = cleaned[month]["accepted"]
        check_interface(accepted, "accepted")
        frames.append(accepted.assign(source=month, rule_rejected=False))
        rejected = cleaned[month]["rejected"]
        only_minutes = rejected.loc[rejected["reasons"].eq("implausible_minutes")].drop(columns="reasons")
        if len(only_minutes):
            frames.append(only_minutes.assign(source=month, rule_rejected=True))
    table = pd.concat(frames, ignore_index=True)
    missing = int(table["minutes"].isna().sum())
    table = table.loc[table["minutes"].notna()].copy()
    table["minutes"] = table["minutes"].astype(float)
    table["hour"] = table["opened_at"].dt.hour.astype(int)
    table["rule_rejected"] = table["rule_rejected"].astype(bool)
    order = {m: i for i, m in enumerate(MONTHS)}
    table = (table.assign(_order=table["source"].map(order))
             .sort_values(["_order", "source_row"], kind="stable")
             .drop(columns="_order").reset_index(drop=True))
    return table, missing

def mad_scores(table):
    """Baseline: robust z-score of minutes inside each request type (median and MAD)."""
    minutes = table["minutes"].astype(float)
    global_mad = float((minutes - minutes.median()).abs().median()) or 1.0
    median = minutes.groupby(table["request_type"]).transform("median")
    mad = (minutes - median).abs().groupby(table["request_type"]).transform("median")
    mad = mad.where(mad > 0, global_mad)      # tiny groups can have MAD = 0
    return ((minutes - median).abs() / (1.4826 * mad)).to_numpy()

def forest_scores(table, contamination=AI_CONTAMINATION, seed=AI_SEED):
    """Isolation Forest on minutes, hour of day and request type; returns (score, flag)."""
    types = pd.get_dummies(table["request_type"], dtype=float)
    types = types.reindex(columns=sorted(types.columns))
    X = np.column_stack([table["minutes"].to_numpy(dtype=float),
                         table["hour"].to_numpy(dtype=float), types.to_numpy()])
    model = IsolationForest(n_estimators=300, contamination=contamination, random_state=seed)
    model.fit(X)
    return -model.score_samples(X), model.predict(X) == -1

model_table, unscored_missing = build_model_table(result["cleaned"])
assert len(model_table) + unscored_missing == (
    int(result["manifest"]["accepted"]) +
    sum(int(result["cleaned"][m]["rejected"]["reasons"].eq("implausible_minutes").sum()) for m in MONTHS))
print("Rows scored:", len(model_table), "| rule-rejected for minutes:", int(model_table["rule_rejected"].sum()),
      "| missing durations not scored:", unscored_missing)

### AI-2 — Score the real pipeline data
- Does each detector catch the rows the fixed rule already rejected?
- Which **accepted** rows do both detectors call unusual? These are the rule's possible blind spots.
- What ceiling would each request type get if it were learned from the data instead of fixed at 480?

In [ ]:
def score_table(table):
    scored = table.copy()
    scored["mad_score"] = mad_scores(scored)
    scored["mad_flag"] = scored["mad_score"] > MAD_THRESHOLD
    scored["forest_score"], scored["forest_flag"] = forest_scores(scored)
    scored["both_flag"] = scored["mad_flag"] & scored["forest_flag"]
    return scored

def suggested_ceilings(scored):
    """Data-driven upper limit per request type, learned from accepted rows only."""
    rows = []
    accepted = scored.loc[~scored["rule_rejected"]]
    for request_type, group in accepted.groupby("request_type"):
        median = float(group["minutes"].median())
        mad = float((group["minutes"] - median).abs().median())
        ceiling = median + MAD_THRESHOLD * 1.4826 * mad
        rows.append({"request_type": request_type, "accepted_rows": int(len(group)),
                     "median_minutes": median, "mad_minutes": mad,
                     "suggested_ceiling": ceiling, "fixed_ceiling": MAX_MINUTES,
                     "accepted_above_suggested": int((group["minutes"] > ceiling).sum())})
    return pd.DataFrame(rows).sort_values("request_type", kind="stable").reset_index(drop=True)

def rule_agreement(scored):
    """How many rule-rejected rows each detector also flags, and how many accepted rows it flags."""
    rule_rows = scored.loc[scored["rule_rejected"]]
    accepted_rows = scored.loc[~scored["rule_rejected"]]
    return pd.DataFrame([
        {"detector": name, "rule_rejected_rows": int(len(rule_rows)),
         "rule_rejected_caught": int(rule_rows[flag].sum()),
         "accepted_rows_flagged": int(accepted_rows[flag].sum())}
        for name, flag in (("MAD baseline", "mad_flag"), ("Isolation Forest", "forest_flag"))])

def review_candidates(scored):
    """Accepted rows that BOTH detectors call unusual, most extreme first."""
    columns = ["source", "source_row", "request_id", "desk_code", "request_type",
               "opened_at", "minutes", "mad_score", "forest_score"]
    picked = scored.loc[~scored["rule_rejected"] & scored["both_flag"], columns]
    return (picked.sort_values(["mad_score", "request_id"], ascending=[False, True], kind="stable")
            .reset_index(drop=True))

scored = score_table(model_table)
agreement = rule_agreement(scored)
ceilings = suggested_ceilings(scored)
review_list = review_candidates(scored)

# Reproducibility: scoring the same table twice must give identical results.
pd.testing.assert_frame_equal(scored, score_table(model_table))
print("Agreement with the fixed rule:"); display(agreement)
print("Suggested ceiling per request type:"); display(ceilings.round(1))
print(f"Accepted rows flagged by BOTH detectors: {len(review_list)} (top 10 shown)")
display(review_list.head(10).round({"minutes": 1, "mad_score": 2, "forest_score": 2}))

### AI-3 — Controlled test with injected anomalies
- The real data has no labels for "wrong duration", so we create a test where the truth is known: in each run, a random **5%** of accepted rows get their duration multiplied by 3–6, **capped at 480** so the fixed rule cannot see them.
- The test starts from accepted rows that the two detectors did **not** both flag in AI-2, so existing oddities are not counted as false alarms.
- 20 seeded runs; report precision, recall and average precision (mean ± std).
- **Read this fairly:** the fixed rule scores zero recall *by construction* (the injected values stay under its ceiling); the meaningful comparison is the model against the MAD baseline. The forest is given the true anomaly share as `contamination`, which is slightly generous to it.

In [ ]:
def injection_experiment(scored, n_runs=20, share=0.05):
    # Start from rows that neither the rule nor both detectors already suspect.
    detector_columns = ["mad_score", "mad_flag", "forest_score", "forest_flag", "both_flag"]
    clean = (scored.loc[~scored["rule_rejected"] & ~scored["both_flag"]]
             .drop(columns=detector_columns).reset_index(drop=True))
    n_inject = max(1, int(round(share * len(clean))))
    rows = []
    for run in range(n_runs):
        rng = np.random.default_rng(AI_SEED + run)
        picked = rng.choice(len(clean), size=n_inject, replace=False)
        truth = np.zeros(len(clean), dtype=bool)
        truth[picked] = True
        trial = clean.copy()
        inflated = trial.loc[picked, "minutes"].to_numpy() * rng.uniform(3, 6, size=n_inject)
        trial.loc[picked, "minutes"] = np.minimum(inflated, MAX_MINUTES)
        mad = mad_scores(trial)
        forest, forest_flag = forest_scores(trial, contamination=n_inject / len(trial), seed=AI_SEED + run)
        rule_flag = ((trial["minutes"] <= 0) | (trial["minutes"] > MAX_MINUTES)).to_numpy()
        for name, flag, score in (("Fixed 480 rule", rule_flag, rule_flag.astype(float)),
                                  ("MAD baseline", mad > MAD_THRESHOLD, mad),
                                  ("Isolation Forest", forest_flag, forest)):
            rows.append({"run": run, "detector": name,
                         "precision": precision_score(truth, flag, zero_division=0),
                         "recall": recall_score(truth, flag, zero_division=0),
                         "average_precision": average_precision_score(truth, score)})
    runs = pd.DataFrame(rows)
    order = ["Fixed 480 rule", "MAD baseline", "Isolation Forest"]
    summary = (runs.groupby("detector")[["precision", "recall", "average_precision"]]
               .agg(["mean", "std"]).reindex(order))
    summary.columns = [f"{metric}_{stat}" for metric, stat in summary.columns]
    return runs, summary.reset_index(), n_inject

def paired_gap(runs, metric="average_precision"):
    """Forest minus baseline on the same runs: mean gap and its standard error."""
    wide = runs.pivot(index="run", columns="detector", values=metric)
    gap = wide["Isolation Forest"] - wide["MAD baseline"]
    return float(gap.mean()), float(gap.std(ddof=1) / np.sqrt(len(gap)))

injection_runs, injection_summary, injected_per_run = injection_experiment(scored)
rerun_runs, _, _ = injection_experiment(scored)
pd.testing.assert_frame_equal(injection_runs, rerun_runs)      # seeded => identical
assert injection_summary.loc[injection_summary["detector"] == "Fixed 480 rule", "recall_mean"].iloc[0] == 0
print(f"Injected anomalies per run: {injected_per_run} | runs: {injection_runs['run'].nunique()}")
ap_gap, ap_gap_se = paired_gap(injection_runs)
display(injection_summary.round(3))
print(f"Average precision, forest minus baseline (paired runs): {ap_gap:+.3f} ± {ap_gap_se:.3f} (standard error)")

### AI-4 — Figures, facts and brief
- Two figures saved to `ai_output/`, three CSVs, and a short brief written **from the measured facts** (the recommended detector is whichever scored higher, not chosen in advance).
- The results ZIP in Google Drive is updated with an `ai_output/` folder; nothing else in it changes.

In [ ]:
def make_ai_figures(scored, ceilings, injection_summary):
    figures = {}

    def save(name, fig):
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=150, facecolor="white", metadata={"Software": "Matplotlib"})
        figures[name] = buf.getvalue()
        plt.show(); plt.close(fig)

    # 1. Durations by request type: what the rule sees vs what the detectors see.
    types = list(ceilings["request_type"])
    accepted = scored.loc[~scored["rule_rejected"]]
    top = max(MAX_MINUTES * 1.08, float(accepted["minutes"].max()) * 1.08)
    off_scale = int(((scored["minutes"] > top) | (scored["minutes"] < 0)).sum())
    rng = np.random.default_rng(AI_SEED)
    fig, ax = plt.subplots(figsize=(10.8, 5.2))
    for i, request_type in enumerate(types):
        rows = scored.loc[scored["request_type"] == request_type]
        x = i + rng.uniform(-0.22, 0.22, size=len(rows))
        normal = (~rows["both_flag"] & ~rows["rule_rejected"]).to_numpy()
        flagged = (rows["both_flag"] & ~rows["rule_rejected"]).to_numpy()
        rejected = rows["rule_rejected"].to_numpy()
        y = rows["minutes"].clip(lower=0, upper=top).to_numpy()
        ax.scatter(x[normal], y[normal], s=16, color="#9DAABD", alpha=.65,
                   label="Accepted, not flagged" if i == 0 else None)
        ax.scatter(x[flagged], y[flagged], s=46, color="#F4AD53", edgecolor="#23344B", linewidth=.8,
                   label="Accepted, flagged by both detectors" if i == 0 else None)
        ax.scatter(x[rejected], y[rejected], s=46, color="#D9534F", marker="x",
                   label="Rejected by the 480 rule" if i == 0 else None)
        limit = float(ceilings.loc[ceilings["request_type"] == request_type, "suggested_ceiling"].iloc[0])
        ax.hlines(limit, i - 0.34, i + 0.34, color="#4F63D9", linewidth=2.4,
                  label="Suggested ceiling for this type" if i == 0 else None)
    ax.axhline(MAX_MINUTES, color="#23344B", linestyle="--", linewidth=1.2, label="Fixed ceiling (480)")
    ax.set_xticks(range(len(types)), types)
    ax.set_ylim(0, top)
    ax.set(xlabel="Request type", ylabel="Handling minutes")
    ax.set_title("AI Project | Fixed ceiling vs learned ceiling per request type", weight="bold", loc="left", pad=14)
    if off_scale:
        ax.text(1.0, 1.02, f"{off_scale} rejected values outside the axis are drawn at its edge",
                transform=ax.transAxes, ha="right", va="bottom", fontsize=9, color="#23344B")
    ax.grid(axis="y", alpha=.19); ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(ncol=3, loc="lower center", bbox_to_anchor=(.5, -0.32), frameon=False, fontsize=9)
    fig.subplots_adjust(left=.08, right=.97, top=.88, bottom=.27)
    save("durations_and_ceilings.png", fig)

    # 2. Injection test: mean and spread over the seeded runs.
    metrics = [("precision", "Precision"), ("recall", "Recall"), ("average_precision", "Average precision")]
    colors = {"Fixed 480 rule": "#9DAABD", "MAD baseline": "#20B9B0", "Isolation Forest": "#4F63D9"}
    x = np.arange(len(metrics)); width = 0.26
    fig, ax = plt.subplots(figsize=(10, 5.0))
    for j, row in enumerate(injection_summary.to_dict("records")):
        means = [row[f"{key}_mean"] for key, _ in metrics]
        stds = [row[f"{key}_std"] for key, _ in metrics]
        bars = ax.bar(x + (j - 1) * width, means, width, yerr=stds, capsize=3,
                      color=colors[row["detector"]], label=row["detector"])
        ax.bar_label(bars, labels=[f"{value:.2f}" for value in means], padding=3, fontsize=9)
    ax.set_xticks(x, [label for _, label in metrics])
    ax.set_ylim(0, 1.12)
    ax.set_ylabel("Score (mean of seeded runs, bars = std)")
    ax.set_title("AI Project | Finding injected anomalies hidden under the 480 ceiling",
                 weight="bold", loc="left", pad=14)
    ax.grid(axis="y", alpha=.19); ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(ncol=3, loc="lower center", bbox_to_anchor=(.5, -0.2), frameon=False)
    fig.subplots_adjust(left=.09, right=.97, top=.88, bottom=.18)
    save("injection_test.png", fig)
    return figures

def ai_brief_facts(scored, agreement, ceilings, review_list, injection_summary, unscored_missing, ap_gap, ap_gap_se):
    by_name = injection_summary.set_index("detector")
    mad_ap = float(by_name.loc["MAD baseline", "average_precision_mean"])
    forest_ap = float(by_name.loc["Isolation Forest", "average_precision_mean"])
    caught = agreement.set_index("detector")["rule_rejected_caught"]
    facts = {"rows_scored": int(len(scored)), "unscored_missing": int(unscored_missing),
             "rule_rejected": int(scored["rule_rejected"].sum()),
             "rule_rejected_caught_mad": int(caught["MAD baseline"]),
             "rule_rejected_caught_forest": int(caught["Isolation Forest"]),
             "accepted_flagged_by_both": int(len(review_list)),
             "lowest_ceiling": float(ceilings["suggested_ceiling"].min()),
             "highest_ceiling": float(ceilings["suggested_ceiling"].max()),
             "mad_recall": float(by_name.loc["MAD baseline", "recall_mean"]),
             "forest_recall": float(by_name.loc["Isolation Forest", "recall_mean"]),
             "mad_average_precision": mad_ap, "forest_average_precision": forest_ap,
             "ap_gap_forest_minus_mad": float(ap_gap), "ap_gap_standard_error": float(ap_gap_se)}
    # A gap smaller than two standard errors is treated as no difference.
    if abs(ap_gap) <= 2 * ap_gap_se:
        facts["better_detector"] = "no clear difference"
        verdict = ("The two detectors ranked the injected anomalies about equally well, so the simpler per-type "
                   "baseline is the better choice: it is easier to explain and needs no model.")
    elif ap_gap > 0:
        facts["better_detector"] = "Isolation Forest"
        verdict = ("The Isolation Forest ranked the injected anomalies better than the simple baseline, "
                   "so the model adds value beyond a per-type statistic.")
    else:
        facts["better_detector"] = "MAD baseline"
        verdict = ("The simple per-type baseline ranked the injected anomalies better than the Isolation Forest, "
                   "so the more complex model is not justified on this dataset.")
    brief = (
        f"This extension scored {facts['rows_scored']} requests with known durations; {facts['unscored_missing']} with missing "
        f"durations were left unscored rather than guessed. The fixed 480-minute rule rejected {facts['rule_rejected']} of the "
        f"scored rows; the MAD baseline also flagged {facts['rule_rejected_caught_mad']} of them and the Isolation Forest "
        f"{facts['rule_rejected_caught_forest']}. Both detectors agree that {facts['accepted_flagged_by_both']} accepted requests "
        f"look unusual for their request type, although every one passed the fixed rule. Ceilings learned per request type range "
        f"from {facts['lowest_ceiling']:.0f} to {facts['highest_ceiling']:.0f} minutes, against one fixed value of 480. In a "
        f"controlled test with anomalies hidden below the fixed ceiling, the fixed rule found none by design, the baseline "
        f"recalled {facts['mad_recall']:.0%} (average precision {facts['mad_average_precision']:.2f}) and the forest "
        f"{facts['forest_recall']:.0%} ({facts['forest_average_precision']:.2f}). {verdict} "
        "Two limitations matter: the data is synthetic, and the injected anomalies are our own design, which favours "
        "detectors that compare a request with its own type. We recommend sending the flagged accepted requests to the data "
        "owner for review instead of rejecting them automatically."
    )
    assert 100 <= len(brief.split()) <= 220, len(brief.split())
    return facts, brief

ai_figures = make_ai_figures(scored, ceilings, injection_summary)
ai_facts, AI_BRIEF = ai_brief_facts(scored, agreement, ceilings, review_list, injection_summary, unscored_missing, ap_gap, ap_gap_se)

def write_ai_outputs(out_dir, scored, ceilings, injection_summary, brief, figures):
    """Write the AI project's files in stable order; returns the file names."""
    columns = ["source", "source_row", "request_id", "desk_code", "request_type", "opened_at", "minutes",
               "rule_rejected", "mad_score", "mad_flag", "forest_score", "forest_flag", "both_flag"]
    files = {"anomaly_scores.csv": _csv_bytes(scored[columns]),
             "suggested_ceilings.csv": _csv_bytes(ceilings),
             "injection_test_summary.csv": _csv_bytes(injection_summary),
             "ai_brief.txt": (brief + "\n").encode("utf-8"), **figures}
    destination = Path(out_dir)
    destination.mkdir(parents=True, exist_ok=True)
    for name in sorted(files):
        (destination / name).write_bytes(files[name])
    return sorted(files)

write_ai_outputs(AI_DIR, scored, ceilings, injection_summary, AI_BRIEF, ai_figures)

# The mandatory outputs must be exactly as the pipeline wrote them.
for name, digest in result["manifest"]["files_sha256"].items():
    assert hashlib.sha256((OUTPUT_DIR / name).read_bytes()).hexdigest() == digest, name

# Add ai_output/ to the results ZIP without touching its other entries.
with zipfile.ZipFile(results_zip) as old:
    kept = {name: old.read(name) for name in old.namelist() if not name.startswith("ai_output/")}
with zipfile.ZipFile(results_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name, payload in kept.items():
        archive.writestr(name, payload)
    for p in sorted(AI_DIR.iterdir()):
        archive.write(p, arcname=f"ai_output/{p.name}")

print("AI facts:", json.dumps(ai_facts, indent=2))
print("\nAI brief (", len(AI_BRIEF.split()), "words):\n", AI_BRIEF)
print("\nAI outputs:", sorted(p.name for p in AI_DIR.iterdir()))
print("PASS: seeded and repeatable; mandatory outputs unchanged; ai_output added to", results_zip.name)